# 04 — Temporal Audit & Candidate t0

**Objetivo:** reconstruir la semántica temporal y evaluar la viabilidad del timestamp de decisión.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Preguntas
- ¿Qué fechas existen y qué cobertura tienen?
- ¿Qué rangos temporales cubre cada fuente?
- ¿Qué campos no parsean correctamente?
- ¿Se cumplen órdenes lógicos básicos de fechas?
- ¿`fecha_de_firma` es un `t0` operativo razonable o debe cambiarse?

Este notebook es crítico: una columna presente hoy no necesariamente existía en el momento histórico de predicción.

In [1]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}

{'processes': 'SECOP_II_-_Procesos_de_Contratación_20260921.csv',
 'contracts': 'SECOP_II_-_Contratos_Electrónicos_20260921.csv',
 'bidders': 'Proponentes_por_Proceso_SECOP_II_20260921.csv',
 'suppliers': 'SECOP_II_-_Proveedores_Registrados_20260921.csv',
 'additions': 'SECOP_II_-_Adiciones_20260921.csv',
 'sanctions': 'SECOPII_-_Multas_y_Sanciones_20260921.csv',
 'payments': 'SECOP_II_-_Plan_de_pagos_20260921.csv'}

In [2]:
from argos.common.reports import write_report
from argos.profiling.temporal import (
    audit_all_temporal,
    audit_order_rule,
    temporal_format_distribution,
    temporal_parse_failures,
)

cfg = PROJECT_CONFIG["profiling"]
temporal_cfg = PROJECT_CONFIG["temporal"]

temporal_profile = audit_all_temporal(
    FILES,
    sample_size=cfg["csv_sample_size"],
    date_order=temporal_cfg["date_order"],
    plausible_year_min=temporal_cfg["plausible_year_min"],
    plausible_year_max=temporal_cfg["plausible_year_max"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    temporal_profile,
    paths.reports / "profiling" / "temporal_profile.parquet",
)

temporal_profile.sort_values(["dataset", "column"])


,dataset,column,normalized_column,row_count,raw_nonnull_count,coverage,parsed_count,unparsed_count,parse_success_rate_nonnull,plausible_parsed_count,plausible_rate_parsed,min_timestamp,max_timestamp,raw_min_timestamp,raw_max_timestamp,date_order,plausible_year_min,plausible_year_max
19,additions,FechaRegistro,fecharegistro,28378604,28378604,1.000000,28378604,0,1.0,28378604,1.0,2015-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00,2015-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00,mdy,2000,2100
17,bidders,Fecha Publicación,fecha_publicacion,2316965,2300785,0.993017,2300785,0,1.0,2300785,1.0,2015-02-14 00:00:00-05:00,2026-09-19 00:00:00-05:00,2015-02-14 00:00:00-05:00,2026-09-19 00:00:00-05:00,mdy,2000,2100
15,contracts,Fecha Fin Liquidacion,fecha_fin_liquidacion,6066730,662134,0.109142,0,662134,0.0,0,NaN,NaT,NaT,NaT,NaT,mdy,2000,2100
14,contracts,Fecha Inicio Liquidacion,fecha_inicio_liquidacion,6066730,662180,0.109149,0,662180,0.0,0,NaN,NaT,NaT,NaT,NaT,mdy,2000,2100
13,contracts,Fecha de Fin del Contrato,fecha_de_fin_del_contrato,6066730,6012117,0.990998,6012117,0,1.0,6012080,0.999994,2002-10-15 00:00:00-05:00,2100-07-28 00:00:00-05:00,0025-10-13 23:52:32-04:56:16,8201-12-21 00:00:00-05:00,mdy,2000,2100
11,contracts,Fecha de Firma,fecha_de_firma,6066730,5640278,0.929706,5640278,0,1.0,5640278,1.0,2015-06-11 00:00:00-05:00,2026-09-19 00:00:00-05:00,2015-06-11 00:00:00-05:00,2026-09-19 00:00:00-05:00,mdy,2000,2100
12,contracts,Fecha de Inicio del Contrato,fecha_de_inicio_del_contrato,6066730,5597819,0.922708,5597819,0,1.0,5597812,0.999999,2002-06-22 00:00:00-05:00,2035-11-07 00:00:00-05:00,0019-01-31 23:52:32-04:56:16,2035-11-07 00:00:00-05:00,mdy,2000,2100
16,contracts,Fecha de notificación de prorrogación,fecha_de_notificacion_de_prorrogacion,6066730,581120,0.095788,0,581120,0.0,0,NaN,NaT,NaT,NaT,NaT,mdy,2000,2100
25,payments,Fecha de Entrega original,fecha_de_entrega_original,1000,979,0.979000,0,979,0.0,0,None,NaT,NaT,NaT,NaT,mdy,2000,2100
21,payments,Fecha de emision,fecha_de_emision,1000,979,0.979000,0,979,0.0,0,None,NaT,NaT,NaT,NaT,mdy,2000,2100


In [3]:
# Re-evaluate known ordering constraints with the explicit SECOP parser.
rules = [
    ("fecha_de_firma", "fecha_de_inicio_del_contrato"),
    ("fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato"),
    ("fecha_inicio_liquidacion", "fecha_fin_liquidacion"),
]

temporal_rules = pd.DataFrame([
    audit_order_rule(
        "contracts",
        FILES["contracts"],
        earlier,
        later,
        sample_size=cfg["csv_sample_size"],
        date_order=temporal_cfg["date_order"],
        temp_directory=paths.root / ".cache" / "duckdb",
    )
    for earlier, later in rules
])

write_report(
    temporal_rules,
    paths.reports / "profiling" / "temporal_rules.parquet",
)

temporal_rules


,dataset,rule,status,earlier_raw_column,later_raw_column,comparable_rows,failures,failure_rate
0,contracts,fecha_de_firma <= fecha_de_inicio_del_contrato,analyzed,Fecha de Firma,Fecha de Inicio del Contrato,5561966,210358,0.037821
1,contracts,fecha_de_inicio_del_contrato <= fecha_de_fin_del_contrato,analyzed,Fecha de Inicio del Contrato,Fecha de Fin del Contrato,5597800,800,0.000143
2,contracts,fecha_inicio_liquidacion <= fecha_fin_liquidacion,analyzed,Fecha Inicio Liquidacion,Fecha Fin Liquidacion,0,0,NaN


In [4]:
# Gate evidence for t0 and the addition-event timestamp.
gate_temporal_columns = [
    ("contracts", "fecha_de_firma"),
    ("contracts", "fecha_de_inicio_del_contrato"),
    ("additions", "fecharegistro"),
    ("sanctions", "fecha_evento"),
]

temporal_gate_view = temporal_profile.loc[
    temporal_profile.apply(
        lambda row: (row["dataset"], row["normalized_column"]) in gate_temporal_columns,
        axis=1,
    ),
    [
        "dataset", "column", "coverage", "parse_success_rate_nonnull",
        "plausible_rate_parsed", "min_timestamp", "max_timestamp",
        "raw_min_timestamp", "raw_max_timestamp",
    ],
]

temporal_gate_view


,dataset,column,coverage,parse_success_rate_nonnull,plausible_rate_parsed,min_timestamp,max_timestamp,raw_min_timestamp,raw_max_timestamp
11,contracts,Fecha de Firma,0.929706,1.0,1.0,2015-06-11 00:00:00-05:00,2026-09-19 00:00:00-05:00,2015-06-11 00:00:00-05:00,2026-09-19 00:00:00-05:00
12,contracts,Fecha de Inicio del Contrato,0.922708,1.0,0.999999,2002-06-22 00:00:00-05:00,2035-11-07 00:00:00-05:00,0019-01-31 23:52:32-04:56:16,2035-11-07 00:00:00-05:00
19,additions,FechaRegistro,1.000000,1.0,1.0,2015-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00,2015-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00
20,sanctions,Fecha Evento,0.902174,0.0,None,NaT,NaT,NaT,NaT


## Decisión
No congeles `t0` por intuición. Documenta por qué el timestamp elegido representa el instante de decisión y qué información era realmente observable entonces.

In [5]:
# Make parser behavior auditable for the two timestamps that define the target.
format_frames = [
    temporal_format_distribution(
        dataset,
        FILES[dataset],
        column,
        sample_size=cfg["csv_sample_size"],
        date_order=temporal_cfg["date_order"],
        temp_directory=paths.root / ".cache" / "duckdb",
    )
    for dataset, column in [
        ("contracts", "fecha_de_firma"),
        ("additions", "fecharegistro"),
    ]
]
temporal_formats = pd.concat(format_frames, ignore_index=True)
write_report(
    temporal_formats,
    paths.reports / "profiling" / "temporal_format_distribution.parquet",
)
temporal_formats


,dataset,column,normalized_column,parse_format,rows
0,contracts,Fecha de Firma,fecha_de_firma,mdy_date,5640278
1,additions,FechaRegistro,fecharegistro,mdy_date,28378604


In [6]:
failure_frames = [
    temporal_parse_failures(
        dataset,
        FILES[dataset],
        column,
        sample_size=cfg["csv_sample_size"],
        date_order=temporal_cfg["date_order"],
        limit=25,
        temp_directory=paths.root / ".cache" / "duckdb",
    )
    for dataset, column in [
        ("contracts", "fecha_de_firma"),
        ("additions", "fecharegistro"),
    ]
]
temporal_failures = pd.concat(failure_frames, ignore_index=True)
write_report(
    temporal_failures,
    paths.reports / "profiling" / "temporal_parse_failures.parquet",
)
temporal_failures


,dataset,column,normalized_column,raw_value,rows
